In [2]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['ChileTrabajos'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [3]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient, UpdateOne
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Benjamin Ramirez"
META_DATOS = 500 # Agregado para cumplir con tu entrega
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    driver.get("https://www.chiletrabajos.cl/encuentra-un-empleo?2=informatica")
    
    nivel_pagina = 1

    # Cambiamos el limite por la cantidad de datos para asegurar los 500
    while len(datos_finales) < META_DATOS:
        print(f"--- Procesando Pagina {nivel_pagina} (Datos capturados hasta ahora: {len(datos_finales)}) ---")

        # Espera fija para que la página termine de cargar
        time.sleep(10)

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[class*='job-item']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[class*='job-item']"
        )

        for bloque in bloques:
            if len(datos_finales) >= META_DATOS:
                break
                
            try:
                # 1. Título del cargo y 2. Empresa
                titulo = bloque.find_element(By.TAG_NAME, "h2").text.strip()
                try:
                    empresa = bloque.find_element(By.TAG_NAME, "h3").text.strip()
                except:
                    empresa = "No especificado"

                # Leemos todo el texto de la tarjeta para buscar las demás etiquetas
                texto_bloque = bloque.text.lower()
                
                # 3. Modalidad de trabajo
                if "teletrabajo" in texto_bloque or "remoto" in texto_bloque:
                    modalidad = "Teletrabajo / Remoto"
                elif "hibrido" in texto_bloque or "híbrido" in texto_bloque:
                    modalidad = "Hibrido"
                else:
                    modalidad = "Presencial / No especificado"

                # 4. Tipo de horario
                if "part time" in texto_bloque or "part-time" in texto_bloque:
                    horario = "Part-time"
                else:
                    horario = "Full-time / No especificado"

                # 5. Tipo de moneda
                if "usd" in texto_bloque:
                    moneda = "USD (Dolares)"
                else:
                    moneda = "CLP (Pesos Chilenos) / Oculto"

                # Se guardan las 8 etiquetas
                datos_finales.append({
                    "titulo_cargo": titulo,
                    "empresa": empresa,
                    "categoria_empleo": NOMBRE_GRUPO,
                    "modalidad_trabajo": modalidad,
                    "pais": "Chile",
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "tipo_moneda": moneda,
                    "tipo_horario": horario
                })
            except:
                continue

        if len(datos_finales) >= META_DATOS:
            break

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.PARTIAL_LINK_TEXT, "Siguiente")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
            nivel_pagina += 1
        except:
            print("No se encontro el boton siguiente o ya es la ultima pagina.")
            break

    print(f"Extraccion terminada: {len(datos_finales)} ofertas de empleo.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    # Cierra el navegador solo si logró abrirse
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["ChileTrabajos"]

    if datos_finales:
        operaciones = []
        for d in datos_finales:
            # Filtra por titulo y empresa para no duplicar si corres el script de nuevo
            filtro = {"titulo_cargo": d["titulo_cargo"], "empresa": d["empresa"]}
            nuevos_valores = {"$set": d}
            operaciones.append(UpdateOne(filtro, nuevos_valores, upsert=True))

        resultado = coleccion.bulk_write(operaciones)
        print(f"Datos cargados en MongoDB correctamente. Nuevos: {resultado.upserted_count}, Actualizados: {resultado.modified_count}.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")

Limpieza de procesos y temporales completada.
Error en Selenium: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location

No hay datos para guardar.
